In [1]:
import h5py
import numpy as np

# File path
file_path = r"C:\Users\risha\Desktop\pcm_project\pcm_2d_dataset.h5"

f = h5py.File(file_path, "r")

T  = f["T"][:]
lf = f["lf"][:]
ux = f["ux"][:]
uy = f["uy"][:]

t = f["t"][:]
x = f["x"][:]
y = f["y"][:]
params = f["params"][:]

f.close()

print("✅ Loaded:", T.shape)
print(f"Cases: {T.shape[0]}, Time steps: {T.shape[1]}, Grid: {T.shape[2]}x{T.shape[3]}")

✅ Loaded: (8, 40, 25, 25)
Cases: 8, Time steps: 40, Grid: 25x25


In [2]:
print("Preparing data (this may take a minute)...")

X_list = []
Y_list = []

for c in range(T.shape[0]):
    T_hot = params[c, 0]
    
    for ti in range(T.shape[1]):
        for j in range(T.shape[2]):
            for i in range(T.shape[3]):
                X_list.append([x[i], y[j], t[ti], T_hot])
                Y_list.append([T[c, ti, j, i], lf[c, ti, j, i], 
                              ux[c, ti, j, i], uy[c, ti, j, i]])

X = np.array(X_list, dtype=np.float32)
Y = np.array(Y_list, dtype=np.float32)

print(f"✅ X shape: {X.shape}")
print(f"✅ Y shape: {Y.shape}")
print(f"Total samples: {X.shape[0]:,}")

Preparing data (this may take a minute)...
✅ X shape: (200000, 4)
✅ Y shape: (200000, 4)
Total samples: 200,000


In [3]:
# Save normalization parameters for later use
X_mean = X.mean(axis=0)
X_std  = X.std(axis=0) + 1e-6

Y_mean = Y.mean(axis=0)
Y_std  = Y.std(axis=0) + 1e-6

X = (X - X_mean) / X_std
Y = (Y - Y_mean) / Y_std

print("✅ Data normalized")
print(f"X range: [{X.min():.2f}, {X.max():.2f}]")
print(f"Y range: [{Y.min():.2f}, {Y.max():.2f}]")
print(f"\nY_std per output: {Y_std}")
print(f"LF normalized range: [{Y[:, 1].min():.2f}, {Y[:, 1].max():.2f}]")

✅ Data normalized
X range: [-1.69, 1.69]
Y range: [-11.13, 17.80]

Y_std per output: [8.7390242e+00 3.7840772e-01 4.1863223e-04 4.2479348e-04]
LF normalized range: [-1.54, 1.11]


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

print(f"✅ Train samples: {X_train.shape[0]:,}")
print(f"✅ Val samples: {X_val.shape[0]:,}")

✅ Train samples: 160,000
✅ Val samples: 40,000


In [5]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {device}")

if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')

X_train = torch.as_tensor(X_train, dtype=torch.float32).to(device)
Y_train = torch.as_tensor(Y_train, dtype=torch.float32).to(device)
X_val = torch.as_tensor(X_val, dtype=torch.float32).to(device)
Y_val = torch.as_tensor(Y_val, dtype=torch.float32).to(device)

print(f"✅ Data on {device}")

✅ Device: cuda
   GPU: NVIDIA GeForce RTX 4050 Laptop GPU
✅ Data on cuda


In [10]:
class SirenLayer(nn.Module):
    def __init__(self, in_f, out_f, w0=30.0, is_first=False):
        super().__init__()
        self.in_f = in_f
        self.w0 = w0
        self.linear = nn.Linear(in_f, out_f)
        self.is_first = is_first
        self.init_weights()
    
    def init_weights(self):
        with torch.no_grad():
            if self.is_first:
                self.linear.weight.uniform_(-1 / self.in_f, 1 / self.in_f)
            else:
                self.linear.weight.uniform_(-np.sqrt(6 / self.in_f) / self.w0, 
                                           np.sqrt(6 / self.in_f) / self.w0)
    
    def forward(self, x):
        return torch.sin(self.w0 * self.linear(x))

class PCM_SIREN(nn.Module):
    def __init__(self, hidden=512, n_layers=8):
        super().__init__()
        
        self.net = nn.ModuleList()
        self.net.append(SirenLayer(4, hidden, is_first=True))
        
        for _ in range(n_layers):
            self.net.append(SirenLayer(hidden, hidden))
        
        self.final = nn.Linear(hidden, 4)
        
        with torch.no_grad():
            self.final.weight.uniform_(-np.sqrt(6 / hidden) / 30.0, 
                                       np.sqrt(6 / hidden) / 30.0)
    
    def forward(self, x):
        for layer in self.net:
            x = layer(x)
        return self.final(x)

print("✅ Simple SIREN model (NO constraints)")

✅ Simple SIREN model (NO constraints)


In [7]:
train_ds = TensorDataset(X_train, Y_train)
val_ds = TensorDataset(X_val, Y_val)

train_dl = DataLoader(train_ds, batch_size=8192, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=8192)

print(f"✅ DataLoader ready (batch=8192)")

✅ DataLoader ready (batch=8192)


In [19]:
import torch.nn.functional as F
from torch.amp import autocast, GradScaler

def ensemble_from_best():
    print(f"\n{'='*70}")
    print(f"ENSEMBLE FROM BEST MODEL - GUARANTEED ≥0.162")
    print(f"{'='*70}\n")
    
    models = []
    
    # Load best model 5 times, fine-tune each differently
    for i in range(1, 6):
        print(f"{'='*60}")
        print(f"Creating Model {i}/5 (fine-tuned from best)")
        print(f"{'='*60}")
        
        # Load best model
        model = PCM_SIREN(hidden=512, n_layers=8).to(device)
        model.load_state_dict(torch.load("best_aggressive.pt", weights_only=False))
        
        # Verify it starts from 0.162
        model.eval()
        with torch.no_grad():
            val = 0
            for X_b, Y_b in val_dl:
                val += F.mse_loss(model(X_b), Y_b).item()
            val /= len(val_dl)
        print(f"  Starting from: {val:.5f}")
        
        # Fine-tune with different strategies
        lrs = [5e-6, 8e-6, 1e-5, 1.2e-5, 1.5e-5]  # Different LRs
        dropouts = [0.0, 0.0, 0.0, 0.0, 0.0]
        
        optimizer = torch.optim.AdamW(model.parameters(), lr=lrs[i-1], weight_decay=1e-6)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200, eta_min=1e-8)
        scaler = GradScaler('cuda')
        
        best_this_model = val
        
        # Fine-tune for 200 epochs
        for epoch in range(1, 201):
            model.train()
            
            for X_b, Y_b in train_dl:
                optimizer.zero_grad()
                
                with autocast('cuda'):
                    pred = model(X_b)
                    
                    # Velocity-focused loss
                    loss = (
                        1.0 * F.mse_loss(pred[:, 0], Y_b[:, 0]) +
                        1.0 * F.mse_loss(pred[:, 1], Y_b[:, 1]) +
                        4.0 * F.mse_loss(pred[:, 2], Y_b[:, 2]) +
                        4.0 * F.mse_loss(pred[:, 3], Y_b[:, 3])
                    ) / 10.0
                
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 0.3)
                scaler.step(optimizer)
                scaler.update()
            
            scheduler.step()
            
            # Check every 20 epochs
            if epoch % 20 == 0:
                model.eval()
                val_loss = 0
                with torch.no_grad():
                    for X_b, Y_b in val_dl:
                        val_loss += F.mse_loss(model(X_b), Y_b).item()
                val_loss /= len(val_dl)
                
                if val_loss < best_this_model:
                    best_this_model = val_loss
                    print(f"  E{epoch:03d} | Val: {val_loss:.5f} ✅")
                else:
                    print(f"  E{epoch:03d} | Val: {val_loss:.5f}")
        
        print(f"  Final: {best_this_model:.5f}\n")
        models.append(model)
    
    # ENSEMBLE EVALUATION
    print(f"\n{'='*70}")
    print(f"ENSEMBLE RESULTS")
    print(f"{'='*70}\n")
    
    print("Individual fine-tuned models:")
    for idx, m in enumerate(models, 1):
        m.eval()
        v = 0
        with torch.no_grad():
            for X_b, Y_b in val_dl:
                v += F.mse_loss(m(X_b), Y_b).item()
        v /= len(val_dl)
        print(f"  Model {idx}: {v:.5f}")
    
    # Ensemble
    ens_val = 0
    ens_out = torch.zeros(4)
    with torch.no_grad():
        for X_b, Y_b in val_dl:
            preds = torch.stack([m(X_b) for m in models])
            ens_pred = preds.mean(dim=0)
            ens_val += F.mse_loss(ens_pred, Y_b).item()
            for i in range(4):
                ens_out[i] += F.mse_loss(ens_pred[:, i].cpu(), Y_b[:, i].cpu()).item()
    
    ens_val /= len(val_dl)
    ens_out /= len(val_dl)
    
    print(f"\n{'='*70}")
    print(f"📊 ENSEMBLE: {ens_val:.5f}")
    print(f"{'='*70}")
    print(f"Baseline: 0.16244")
    print(f"Ensemble: {ens_val:.5f}")
    
    if ens_val < 0.16244:
        improvement = (0.16244 - ens_val) / 0.16244 * 100
        print(f"✅ Improved by {improvement:.2f}%")
    
    print(f"\nPer-output:")
    print(f"  T:  {ens_out[0]:.5f}")
    print(f"  lf: {ens_out[1]:.5f}")
    print(f"  ux: {ens_out[2]:.5f}")
    print(f"  uy: {ens_out[3]:.5f}")
    
    if ens_val < 0.1:
        print(f"\n🎉 <0.1 ACHIEVED!")
    else:
        print(f"\nGap to 0.1: {ens_val - 0.1:.5f}")
    
    return models, ens_val

models, final = ensemble_from_best()


ENSEMBLE FROM BEST MODEL - GUARANTEED ≥0.162

Creating Model 1/5 (fine-tuned from best)
  Starting from: 0.16244
  E020 | Val: 0.16200 ✅
  E040 | Val: 0.16151 ✅
  E060 | Val: 0.16136 ✅
  E080 | Val: 0.16134 ✅
  E100 | Val: 0.16124 ✅
  E120 | Val: 0.16117 ✅
  E140 | Val: 0.16115 ✅
  E160 | Val: 0.16121
  E180 | Val: 0.16117
  E200 | Val: 0.16123
  Final: 0.16115

Creating Model 2/5 (fine-tuned from best)
  Starting from: 0.16244
  E020 | Val: 0.16161 ✅
  E040 | Val: 0.16120 ✅
  E060 | Val: 0.16100 ✅
  E080 | Val: 0.16092 ✅
  E100 | Val: 0.16072 ✅
  E120 | Val: 0.16060 ✅
  E140 | Val: 0.16056 ✅
  E160 | Val: 0.16051 ✅
  E180 | Val: 0.16054
  E200 | Val: 0.16052
  Final: 0.16051

Creating Model 3/5 (fine-tuned from best)
  Starting from: 0.16244
  E020 | Val: 0.16064 ✅
  E040 | Val: 0.16042 ✅
  E060 | Val: 0.16002 ✅
  E080 | Val: 0.15993 ✅
  E100 | Val: 0.16002
  E120 | Val: 0.15988 ✅
  E140 | Val: 0.15973 ✅
  E160 | Val: 0.15969 ✅
  E180 | Val: 0.15971
  E200 | Val: 0.15974
  Final: 0.1